In [16]:
"""
standard imports
"""

import uproot
import awkward as ak
import argparse
import logging
from tqdm import tqdm
from pathlib import Path
import os
import numpy as np

import h5py
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from livelossplot import PlotLosses



"""
torch imports
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import IterableDataset
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast





"""
module imports
"""

import sys

# adjust path to project properly

#PATH_TO_PROJECT = '/global/cfs/cdirs/m4474/aneek/particlemind_aneek'
PATH_TO_PROJECT = '/Users/aneekphys/Documents/Other Research/hep-ssl'

sys.path.append(PATH_TO_PROJECT)


from src.data.dataset import ColliderMLHits, projected_hits
from src.data.coarsening import voxelize_hits
from src.data.augmentation import *


PATH_TO_SPECTER = '/Users/aneekphys/Documents/Other Research/SPECTER/src'
sys.path.append(PATH_TO_SPECTER)

from pyspecter.SPECTER import SPECTER


In [2]:
"""
Loading ColliderML data

"""

CACHE_DIR = '/Users/aneekphys/Downloads'

from colliderml.core import load_tables, collect_tables


## ttbar data
cfg1 = {
    "dataset_id": "CERN/ColliderML-Release-1",
    "channels": "ttbar",
    "pileup": "pu0",
    "objects": ["calo_hits"],
    "split": "train",
    "lazy": False,
    "max_events": 100,
    "data_dir":CACHE_DIR
}
tables1 = load_tables(cfg1)
frames1 = collect_tables(tables1)

calo_hits1 = frames1["calo_hits"][0:100]


## dihiggs data
cfg2 = {
    "dataset_id": "CERN/ColliderML-Release-1",
    "channels": "dihiggs",
    "pileup": "pu0",
    "objects": ["calo_hits"],
    "split": "train",
    "lazy": False,
    "max_events":100,
    "data_dir":CACHE_DIR
}
tables2 = load_tables(cfg2)
frames2 = collect_tables(tables2)

calo_hits2 = frames2["calo_hits"][0:100]



"""
some processing to create iterable datasets:

each instance is a dictionary with --> key 'calo_hit_features' giving hit features (of an entire event) of shape (num_hits, 4) with 
first 3 features (x,y,z) and the last feature being energy E.
"""

hits_ttbar = ColliderMLHits(calo_hits1, "train", shuffle_files=False, train_fraction=1.0)
hits_dihiggs = ColliderMLHits(calo_hits2, "train", shuffle_files=False, train_fraction=1.0)


In [18]:
specter = SPECTER(periodic_indices=[1])

B = 10

i=0

#augment = RandomRotateXY(angle_range=(-np.pi/8, np.pi/8), gaussian = False)

augment = EnergyWhiteNoise(0.2, log=False)

#augment = NoiseXYZ(5)

emds = []

for hits_ in hits_ttbar:
    
    hits = hits_['calo_hit_features']
    
    event = CaloEvent(hits)
    
    event_aug = augment(event)
    
    proj_ = projected_hits(event.hits, grid_size=32, typ='hits')['eta-phi'][:, [2,0,1]]
    proj_[:,0] /= proj_[:,0].sum()
    
    proj_aug = projected_hits(event_aug.hits, grid_size=32, typ='hits')['eta-phi'][:,[2,0,1]]
    proj_aug[:,0] /= proj_aug[:,0].sum()
    
    proj_ = proj_.reshape(1,-1,3)
    proj_aug = proj_aug.reshape(1,-1,3)
    
    #emd = specter.spectralEMD(proj_, proj_aug, metric='cylindrical')
    
    emd = specter.spectralEMD(proj_, proj_aug, metric='mahalanobis', m_mode='event-wise', m_weighting='uniform', m_matrix=None)

    
    emds.append(emd[0])
    
    i+=1
    
    if i >B:
        break
    
print(emds)
    
    
    
    
    
    

Compiling SPECTER model...
Generating test events for tracing ...
Test events generated! Time taken:  3.1419880390167236  seconds.
Compiling spectral representation functions ...
Compilation complete! Time taken:  8.696959972381592  seconds.
[Array(0.000732, dtype=float32), Array(0.00090632, dtype=float32), Array(0.00653022, dtype=float32), Array(0.00764029, dtype=float32), Array(0.00159219, dtype=float32), Array(0.00152124, dtype=float32), Array(0.00184736, dtype=float32), Array(0.00083627, dtype=float32), Array(0.00038189, dtype=float32), Array(0.00043948, dtype=float32), Array(0.00047265, dtype=float32)]


In [13]:
specter = SPECTER(periodic_indices=[1])

B = 50

i=0

emds =[]

for hits_ in hits_ttbar:
    
    hits = hits_['calo_hit_features']
    
    event = CaloEvent(hits)
    
    
    
    proj_ = projected_hits(event.hits, grid_size=64, typ='hits')['eta-phi'][:, [2,0,1]]
    proj_[:,0] /= proj_[:,0].sum()
   
    proj_ = proj_.reshape(1,-1,3)
    
    
    
    for hits_h_ in hits_dihiggs:
        
        hits_h = hits_h_['calo_hit_features']
        
        event_h = CaloEvent(hits_h)
        
        proj_h = projected_hits(event_h.hits, grid_size=64, typ='hits')['eta-phi'][:, [2,0,1]]
        proj_h[:,0] /= proj_h[:,0].sum()

        proj_h = proj_h.reshape(1,-1,3)
        
    
    
        emd = specter.spectralEMD(proj_, proj_h, metric='mahalanobis', m_mode='event-wise', m_weighting='energy', m_matrix=None)
    
        emds.append(emd[0])
        
        break
        
    
    i+=1
    
    if i >B:
        break
    
print(emds)
    
    
    
    


Compiling SPECTER model...
Generating test events for tracing ...
Test events generated! Time taken:  4.092796802520752  seconds.
Compiling spectral representation functions ...
Compilation complete! Time taken:  8.306631803512573  seconds.
[Array(0.005395, dtype=float32), Array(0.00702161, dtype=float32), Array(0.00871404, dtype=float32), Array(0.00558687, dtype=float32), Array(0.003968, dtype=float32), Array(0.00122249, dtype=float32), Array(0.00651194, dtype=float32), Array(0.00974258, dtype=float32), Array(0.00099441, dtype=float32), Array(0.00843014, dtype=float32), Array(0.00251102, dtype=float32), Array(0.00328777, dtype=float32), Array(0.00406129, dtype=float32), Array(0.00125402, dtype=float32), Array(0.00427631, dtype=float32), Array(0.00262801, dtype=float32), Array(0.00324246, dtype=float32), Array(0.00110925, dtype=float32), Array(0.0057534, dtype=float32), Array(0.00810283, dtype=float32), Array(0.00977743, dtype=float32), Array(0.00468257, dtype=float32), Array(0.0064875

In [14]:
np.mean(emds)


np.float32(0.0055545527)

In [15]:
np.std(emds)

np.float32(0.0028515905)

In [35]:
len(emds)

51